# AttnRes VLM Ablation (Colab / CUDA)

Exploratory Full/Block Attention Residuals study on a tiny modular VLM with a **hard TinyShapesVQA** difficulty ladder (levels 1–5).

Open with a GPU runtime and **Run all**. Persistent artifacts go under `MyDrive/SharedColab/attnres_vlm_ablation` and are keyed by **config hash**, so the completed easy-benchmark runs stay intact.

Before the full multi-seed grid, the runner trains one baseline quick probe. If levels 4–5 are still >95% accurate, it automatically bumps difficulty and retries. W&B project: `attnres-next-scale-vlm`.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content
!rm -rf /content/AttnResGPT-next-scale
!git clone https://github.com/AtinChing/AttnResGPT-next-scale.git /content/AttnResGPT-next-scale
%cd /content/AttnResGPT-next-scale
!pip install -q -r requirements.txt
!git rev-parse --short HEAD
!git log -1 --oneline


In [ ]:
import sys
from pathlib import Path

REPO = Path("/content/AttnResGPT-next-scale")
DRIVE_ROOT = Path("/content/drive/MyDrive")
SHARED_COLAB_ROOT = DRIVE_ROOT / "SharedColab"
PROJECT_ROOT = SHARED_COLAB_ROOT / "attnres_vlm_ablation"

SHARED_COLAB_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for name in [
    "source",
    "configs",
    "checkpoints",
    "runs",
    "logs",
    "metrics",
    "summaries",
    "plots",
    "tables",
    "cache",
    "manifests",
]:
    (PROJECT_ROOT / name).mkdir(parents=True, exist_ok=True)

# Drop cached imports so a fresh clone is actually used without Restart runtime.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.vlm.synthetic_vqa import VQATokenizer

tok = VQATokenizer()
assert "triangles" in tok.token_to_id, "Old clone/cache detected: plural shape tokens missing"
print("REPO:", REPO)
print("cloned commit:", (REPO / ".git" / "HEAD").read_text().strip() if False else __import__("subprocess").check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip())
print("DRIVE_ROOT:", DRIVE_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("tokenizer has triangles/circles/squares:", all(t in tok.token_to_id for t in ("triangles", "circles", "squares")))


## Weights & Biases login


In [4]:
import os
import wandb

# Prefer env var / Colab secret. Do not hardcode API keys in the notebook.
try:
    from google.colab import userdata

    if not os.environ.get("WANDB_API_KEY"):
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass

WANDB_ENTITY = "atin5551-uc-davis"
WANDB_PROJECT = "attnres-next-scale-vlm"
WANDB_ENABLED = True
WANDB_MODE = "auto"  # auto | online | offline | disabled
WANDB_RESUME = "allow"  # allow | must | never

wandb.login()
print("W&B project:", WANDB_PROJECT)
print("W&B entity:", WANDB_ENTITY)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: atin5551 (atin5551-uc-davis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B project: attnres-next-scale-vlm
W&B entity: atin5551-uc-davis


## Configuration


In [ ]:
RUN_MODE = "quick"  # "smoke", "quick", or "full"

RESUME = True
FORCE_RESTART = False

RUN_PRIMARY_FULL_GRID = True
RUN_BLOCK_GRID = True

SEEDS = [0, 1, 2]

PRIMARY_VARIANTS = [
    "baseline",
    "encoder_full",
    "decoder_full",
    "both_full",
]

BLOCK_VARIANTS = [
    "encoder_block",
    "decoder_block",
    "both_block",
]

# Hard TinyShapesVQA knobs (new config hash; does not overwrite easy-benchmark runs)
DATASET_VERSION = "hard_v1"
DIFFICULTY_BUMP_LEVEL = 0
SANITY_CHECK_BASELINE = True
SANITY_MAX_BUMPS = 3
SANITY_L45_ACCURACY_CEILING = 0.95

# Training / model knobs (T4-compatible defaults)
BATCH_SIZE = 64
NUM_WORKERS = 2
CHECKPOINT_INTERVAL = 100
EVALUATION_INTERVAL = 1
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.05
MIXED_PRECISION = True
AMP_DTYPE = "auto"  # "auto" | "float16" | "bfloat16"

VISION_D_MODEL = 128
VISION_N_LAYERS = 6
VISION_N_HEADS = 4
VISION_D_FF = 512
DECODER_D_MODEL = 128
DECODER_N_LAYERS = 6
DECODER_N_HEADS = 4
DECODER_D_FF = 512
NUM_BLOCKS = 3
DROPOUT = 0.0

print("RUN_MODE:", RUN_MODE)
print("DATASET_VERSION:", DATASET_VERSION)
print("SANITY_CHECK_BASELINE:", SANITY_CHECK_BASELINE)
print("PRIMARY_VARIANTS:", PRIMARY_VARIANTS)
print("BLOCK_VARIANTS:", BLOCK_VARIANTS)
print("SEEDS:", SEEDS)
print("WANDB_PROJECT:", WANDB_PROJECT)


## CUDA enforcement


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. Select a Colab GPU runtime and rerun. "
        "Training will not fall back to CPU or MPS."
    )

props = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
major, minor = torch.cuda.get_device_capability(0)
selected_amp = "bfloat16" if (AMP_DTYPE == "auto" and major >= 8) else (
    "float16" if AMP_DTYPE == "auto" else AMP_DTYPE
)

print("GPU:", props.name)
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)
print("Total GPU memory bytes:", int(total_bytes))
print("Available GPU memory bytes:", int(free_bytes))
print("Selected AMP dtype:", selected_amp)


## Run experiment


In [ ]:
from src.vlm.ablation.config import AblationExperimentConfig, config_hash, resolve_experiment_config
from src.vlm.ablation.runner import run_ablation_experiment

config = AblationExperimentConfig(
    run_mode=RUN_MODE,
    resume=RESUME,
    force_restart=FORCE_RESTART,
    run_primary_full_grid=RUN_PRIMARY_FULL_GRID,
    run_block_grid=RUN_BLOCK_GRID,
    seeds=list(SEEDS),
    primary_variants=list(PRIMARY_VARIANTS),
    block_variants=list(BLOCK_VARIANTS),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    evaluation_interval=EVALUATION_INTERVAL,
    max_epochs=MAX_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    mixed_precision=MIXED_PRECISION,
    amp_dtype=AMP_DTYPE,
    vision_d_model=VISION_D_MODEL,
    vision_n_layers=VISION_N_LAYERS,
    vision_n_heads=VISION_N_HEADS,
    vision_d_ff=VISION_D_FF,
    decoder_d_model=DECODER_D_MODEL,
    decoder_n_layers=DECODER_N_LAYERS,
    decoder_n_heads=DECODER_N_HEADS,
    decoder_d_ff=DECODER_D_FF,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
    dataset_version=DATASET_VERSION,
    difficulty_bump_level=DIFFICULTY_BUMP_LEVEL,
    sanity_check_baseline=SANITY_CHECK_BASELINE,
    sanity_max_bumps=SANITY_MAX_BUMPS,
    sanity_l45_accuracy_ceiling=SANITY_L45_ACCURACY_CEILING,
    project_root=str(PROJECT_ROOT),
    wandb_enabled=WANDB_ENABLED,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_mode=WANDB_MODE,
    wandb_resume=WANDB_RESUME,
)

preview = resolve_experiment_config(config)
print("Pre-sanity config hash:", config_hash(preview))
print("Dataset version:", preview.dataset_version, "bump:", preview.difficulty_bump_level)

summary = run_ablation_experiment(
    config,
    project_root=PROJECT_ROOT,
    source_root=REPO,
)
summary


## Completion summary


In [ ]:
import json

summary_path = PROJECT_ROOT / "summaries" / "experiment_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

print("Mounted Drive root:", DRIVE_ROOT)
print("Experiment root:", PROJECT_ROOT)
print("Repo clone:", REPO)
print("GPU information:", json.dumps(summary.get("cuda", {}), indent=2))
print("Run mode:", summary.get("run_mode"))
print("Dataset version:", summary.get("dataset_version"))
print("Difficulty bump level:", summary.get("difficulty_bump_level"))
print("Config hash:", summary.get("config_hash"))
print("Sanity gate:", json.dumps(summary.get("sanity_gate", {}), indent=2)[:2000])
print("W&B project:", summary.get("wandb_project"))
print("W&B entity:", summary.get("wandb_entity"))
print("W&B summary run:", summary.get("wandb_summary"))
print("Requested variants:", summary.get("requested_variants"))
print("Completed:", summary.get("completed"))
print("Resumed:", summary.get("resumed"))
print("Failed:", summary.get("failed"))
print("Best by variant:")
for variant, metrics in (summary.get("best_by_variant") or {}).items():
    print(
        f"  {variant}: val={metrics.get('validation_accuracy')} "
        f"test={metrics.get('test_accuracy')} "
        f"L4/L5={ {k: (metrics.get('level_accuracy_test') or {}).get(k) for k in ('4','5')} } "
        f"answer_nll={metrics.get('test_answer_token_nll')} "
        f"best_ckpt={metrics.get('checkpoint_best')}"
    )
print("Aggregate CSV paths:", summary.get("tables"))
print("Plot directory:", summary.get("plots_dir"))
print("Hashed summary:", PROJECT_ROOT / "summaries" / f"experiment_summary_{summary.get('config_hash')}.json")
print("All persistent artifacts under Google Drive:", summary.get("artifacts_under_drive"))
assert str(PROJECT_ROOT).startswith(str(DRIVE_ROOT)), "Project root escaped Drive"
print("Idempotent re-run safe: completed runs are skipped via completed.marker")
print("Easy-benchmark runs remain under their original config-hash directories.")
